# VisionGuard — Fine-Tuning YOLOv8 on Custom Weapon Dataset

This notebook trains a custom YOLOv8 model on the provided Roboflow **Weapon Detection** dataset (`fypweapon/weapon-detection-sckyd`).

### Classes in this dataset:
- `Knife` (Mapped to **Sharp Objects** in VisionGuard)
- `Gun`, `Guns`, `Handgun`, `Rifle` (Mapped to **Weapons** in VisionGuard)

> **Tip:** Running this on Google Colab with a free T4 GPU completes in ~15-20 minutes.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics

In [ ]:
import os
import zipfile

# 3. Extract dataset if running in fresh environment
zip_path = 'data/Weapon Detection.v1-dataset.yolov8.zip'
extract_dir = 'data/weapon-detection'

if os.path.exists(zip_path) and not os.path.exists(extract_dir):
    print('Extracting weapon dataset...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('Extraction complete!')
else:
    print(f'Dataset ready at: {extract_dir}')

In [ ]:
# 4. Train YOLOv8 model
from ultralytics import YOLO

# Using YOLOv8s as base for higher detection accuracy
model = YOLO('yolov8s.pt')

results = model.train(
    data='data/weapon-detection/data.yaml',
    epochs=30,
    imgsz=416,
    batch=16,
    name='weapon_v1',
    project='experiments/runs',
    workers=4,
    save=True,
    exist_ok=True
)

In [ ]:
# 5. Validate model metrics
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")

In [ ]:
# 6. Copy best weights to VisionGuard backend models folder
import shutil

best_weights = 'experiments/runs/weapon_v1/weights/best.pt'
target_path = 'backend/models/weapon_yolo.pt'

if os.path.exists(best_weights):
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    shutil.copy(best_weights, target_path)
    print(f'Model deployed successfully to {target_path}!')
else:
    print('Best weights not found yet. Check training runs directory.')